# Stage 3 & 4 — Matching & Submission Pipeline

This notebook covers the complete final matching pipeline:
1. **Feature Engineering (Stage 3)**: Compute fine-grained RapidFuzz string similarities, numeric/house number overlaps, and address ratios.
2. **Model Training (Stage 4)**: Train a gradient-boosted decision tree (LightGBM) on candidate pairs.
3. **Threshold Optimization**: Tune the decision threshold $\tau$ to maximize **Macro $F_{0.5}$** (penalizing false merges and maximizing singleton credit).
4. **Inference & Submission**: Score test candidates, format `output/matching_results.tsv`, and validate against official rules.
5. **S3 Cloud Backup**: Upload the trained model and final submission directly to Amazon S3.

In [ ]:
# 1. Environment & Path Setup
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import boto3

# Ensure project root in sys.path
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.matching import (
    extract_pair_features,
    MatcherModel,
    train_matcher,
    find_optimal_threshold,
    evaluate_f05,
    MatchingPipeline,
)
from src.matching.pipeline import generate_matching_results, write_submission

print("Environment verified! LightGBM version:", lgb.__version__)

In [ ]:
# 2. Inspect Stage 2 Candidate Pairs
# Load validation candidate pairs generated by Stage 2
val_cand_path = "../dataset/candidates/val_candidate_pairs.parquet"

if os.path.exists(val_cand_path):
    val_cands = pd.read_parquet(val_cand_path)
    print(f"Validation Candidate Pairs: {len(val_cands):,}")
    print(f"Unique S1 Queries: {val_cands['query_id'].nunique():,}")
    print(f"Average candidates per query: {len(val_cands) / val_cands['query_id'].nunique():.1f}")
    display(val_cands.head(5))
else:
    print(f"{val_cand_path} not found. Run Stage 2 blocking first:")
    print("  python ../scripts/run_blocking.py --split val")

In [ ]:
# 3. Run Stage 3 & 4 Training & Threshold Optimization
# Trains LightGBM and tunes decision threshold tau for Macro F_0.5
!python ../scripts/run_matching.py --mode train \
    --train-candidates ../dataset/candidates/val_candidate_pairs.parquet \
    --val-candidates ../dataset/candidates/val_candidate_pairs.parquet \
    --preprocessed-dir ../dataset/preprocessed \
    --model-path ../models/lightgbm_matcher.txt

In [ ]:
# 4. Inspect Feature Importances & Threshold Report
model = MatcherModel.load("../models/lightgbm_matcher.txt")
imp_df = model.get_feature_importances()

print("Top 15 Most Discriminative Features:")
display(imp_df.head(15))

# Load threshold report
report_path = "../dataset/candidates/threshold_tuning_report.json"
if os.path.exists(report_path):
    with open(report_path) as f:
        rep = json.load(f)
    print(f"\nOptimal Threshold (tau*): {rep['best_threshold']:.2f}")
    print(f"Validation Macro F_0.5 Score: {rep['best_macro_f05']:.4f}")

In [ ]:
# 5. Run Test Inference & Generate Final Leaderboard Submission
# Generates output/matching_results.tsv and verifies against validator
!python ../scripts/run_matching.py --mode test \
    --test-candidates ../dataset/candidates/test_candidate_pairs.parquet \
    --test-candidate-tsv ../output/candidate_pairs.tsv \
    --model-path ../models/lightgbm_matcher.txt \
    --preprocessed-dir ../dataset/preprocessed \
    --out-file ../output/matching_results.tsv

In [ ]:
# 6. Validate Submission File Structure
!python ../student_resource/utils/validate_submission.py \
    --matching ../output/matching_results.tsv \
    --candidate ../output/candidate_pairs.tsv \
    --test-dir ../student_resource/dataset/test

In [ ]:
# 7. Backup Trained Model & Submission to S3 Bucket
s3 = boto3.client("s3")
bucket = "amazon-ml-challenge-satyam"

try:
    print(f"Uploading model to s3://{bucket}/models/...")
    s3.upload_file("../models/lightgbm_matcher.txt", bucket, "models/lightgbm_matcher.txt")
    
    print(f"Uploading submission to s3://{bucket}/submissions/...")
    s3.upload_file("../output/matching_results.tsv", bucket, "submissions/matching_results.tsv")
    print("Upload complete! Artifacts safely stored in cloud storage.")
except Exception as e:
    print("S3 upload error:", e)